In [25]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from src.features.build_pipe_fe import build_pipe_fe
from src.features.new_features import TimeFeatureStrategy, AgeCategoryFeatureStrategy, HourCategoryFeatureStrategy
from sklearn.model_selection import train_test_split, GridSearchCV


In [2]:
df_transactions = pd.read_csv('../data/raw/transactions.csv', encoding='utf-8')
df_customers = pd.read_csv('../data/raw/customers.csv', encoding='utf-8')
df = pd.merge(
    df_transactions,
    df_customers,
    left_on='sender_id',
    right_on='customer_id',
    how='left'
)
target = 'fraud'
X = df.drop(columns=[target])
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [3]:
features = [
    TimeFeatureStrategy(),
    AgeCategoryFeatureStrategy(),
    HourCategoryFeatureStrategy()
]

In [4]:
cat_cols = ['hour_category', 'age_category', 'account_type', 'gender', 'device_type']
num_cols = ['amount', 'age']

columns_drop = [
    'transaction_id', 'timestamp', 'sender_id', 'receiver_id',
    'customer_id', 'cpf', 'pix_key', 'hour_date', 'minute_date'
]

In [5]:
pipe_fe = build_pipe_fe(num_cols=num_cols, cat_cols=cat_cols, features=features, columns_drop=columns_drop)
X_resampled, y_resampled = pipe_fe.fit_resample(X_train, y_train)

In [13]:
pipe_test_fe = build_pipe_fe(num_cols=num_cols, cat_cols=cat_cols, features=features, columns_drop=columns_drop)
X_test_fe, y_test_fe = pipe_test_fe.fit_resample(X_test, y_test)

---

## Select Model

Parametros:
- Precision, recall e f1-score

In [26]:
models_params = [
    {
        'name': 'LogisticRegression',
        'model': LogisticRegression(max_iter=1000, class_weight='balanced'),
        'param_grid': {
            'C': [0.01, 0.1, 1, 10],
            'penalty': ['l2'],
            'solver': ['lbfgs', 'liblinear']
        }
    },
    {
        'name': 'RandomForest',
        'model': RandomForestClassifier(n_estimators=100, class_weight='balanced'),
        'param_grid': {
            'n_estimators': [100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5]
        }
    }
]

In [27]:
for mp in models_params:
    print(f"Rodando GridSearch para {mp['name']}...")
    grid = GridSearchCV(mp['model'], mp['param_grid'], cv=5, scoring='f1')
    grid.fit(X_resampled, y_resampled)

    print(f"Melhores parâmetros {mp['name']}: {grid.best_params_}")
    print(f"Melhor F1 {mp['name']}: {grid.best_score_}\n")

Rodando GridSearch para LogisticRegression...
Melhores parâmetros LogisticRegression: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Melhor F1 LogisticRegression: 0.540100491501879

Rodando GridSearch para RandomForest...
Melhores parâmetros RandomForest: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Melhor F1 RandomForest: 0.8832083767848916

